# Investment Timing

T7 used `Sizing` across periods: each period independently picks a size, `effects_per_size` is charged each period as an annualized rate. That works when you can pretend CAPEX is amortized into a yearly fee. Real planning problems often need lumpier semantics: *one boiler, one build, one CAPEX bill*. That's `Investment`.

The Investment object differs from `Sizing` in three ways:

- **One size, one build period** — the optimizer picks ONE size for the whole horizon and ONE period to build in.
- **CAPEX vs OPEX** — separate `effects_*_at_build` (one-time, in the build period) from `effects_*_recurring` (every active period).
- **`prior_size`, `lifetime`, `mandatory`** — describe the existing fleet, asset lifetime, and whether building is required.

Same gas → boiler → workshop heat system as T7, but now with a longer horizon — five periods at four-year gaps — to give the optimizer room to place the build.


In [ ]:
from datetime import datetime, timedelta

from fluxopt import Carrier, Converter, Effect, Flow, Investment, Port, optimize

In [ ]:
n = 24
timesteps = [datetime(2024, 1, 15) + timedelta(hours=h) for h in range(n)]
periods = [2024, 2028, 2032, 2036, 2040]
demand = [10, 10, 8, 8, 8, 12, 25, 60, 70, 75, 75, 70, 65, 60, 55, 50, 45, 40, 30, 25, 20, 15, 12, 10]
peak = max(demand)


def build(boiler_size, **extra):
    return optimize(
        timesteps=timesteps,
        periods=periods,
        carriers=[Carrier('gas'), Carrier('heat')],
        effects=[Effect('cost', unit='EUR')],
        ports=[
            Port(
                'gas_grid',
                imports=[Flow('gas', size=1000, effects_per_flow_hour={'cost': 0.08})],
            ),
            Port(
                'workshop',
                exports=[Flow('heat', size=peak, fixed_relative_profile=[d / peak for d in demand])],
            ),
        ],
        converters=[
            Converter.boiler(
                'boiler',
                thermal_efficiency=0.9,
                fuel_flow=Flow('gas', size=300),
                thermal_flow=Flow('heat', size=boiler_size),
            ),
        ],
        objective_effects='cost',
        **extra,
    )

## 1. From `Sizing` to `Investment`

Drop `Sizing` and pass an `Investment` object instead. The minimum spec — `min_size`, `max_size`, `mandatory=True`, plus the at-build CAPEX — gives you "build exactly once, optimizer picks the period and the size":


In [ ]:
result = build(
    boiler_size=Investment(
        min_size=50,
        max_size=200,
        mandatory=True,
        effects_per_size_at_build={'cost': 1500},
    ),
)
print(f'Built size: {float(result.solution["invest--size"].sel(flow="boiler(heat)").values):.1f} kW')
result.solution['invest--build'].sel(flow='boiler(heat)').to_dataframe(name='built')

**One** size (75 kW — the peak), and **one** build period (2024). With flat weights and constant fuel costs, CAPEX is paid once whenever you build, so any period works — the solver picks the first.

Compare with T7's `Sizing(effects_per_size={'cost': 1500})`: there, 1500/MW was charged in *each* period (~3 × CAPEX over the horizon). Here, 1500/MW is charged in *one* period only — a true CAPEX bill.


## 2. CAPEX + recurring O&M

Real assets cost money to run, not just to build. `effects_per_size_recurring` charges per MW *every active period* — fixed maintenance, insurance, anything that scales with capacity but not with output. With both fields, the optimizer balances "build big once" against "pay recurring forever".


In [ ]:
result = build(
    boiler_size=Investment(
        min_size=50,
        max_size=200,
        mandatory=True,
        effects_per_size_at_build={'cost': 1500},
        effects_per_size_recurring={'cost': 30},  # €30/kW/period maintenance
    ),
)
print(f'Built size: {float(result.solution["invest--size"].sel(flow="boiler(heat)").values):.1f} kW')
result.effect_totals.to_dataframe(name='cost/period').round(2)

CAPEX shows up only in the build period; recurring O&M shows up every period from the build onwards.

`effects_fixed_at_build` and `effects_fixed_recurring` are the analogous fixed-cost variants — one-time and recurring lump sums independent of size. They're useful with `mandatory=False` to charge a connection fee whether you build a 50 kW or a 200 kW boiler.


## 3. Period-varying CAPEX

Costs change over time. Carbon prices rise, technology gets cheaper, supply chains tighten. Pass a list aligned with `periods` to `effects_per_size_at_build` and the optimizer weighs the trade-off when choosing *when* to build.


In [ ]:
# CAPEX rises 8 %/period as material costs climb
capex_by_period = [1500, 1620, 1750, 1890, 2040]

result = build(
    boiler_size=Investment(
        min_size=50,
        max_size=200,
        mandatory=True,
        effects_per_size_at_build={'cost': capex_by_period},
        effects_per_size_recurring={'cost': 30},
    ),
)
df = result.solution['invest--build'].sel(flow='boiler(heat)').to_dataframe(name='built')
df['CAPEX/MW'] = capex_by_period
df

Build in 2024 — locking in the lowest CAPEX. With NPV weights or a rising fuel price, this trade-off can flip; the discount on future CAPEX may outweigh the technology learning curve.


## 4. `prior_size`: existing capacity

Most planning starts from a non-empty grid. `prior_size=X` says "X units of capacity exist from period 0, no CAPEX charged for it." Combined with `mandatory=False`, the optimizer can skip the new build entirely if the prior already covers demand.


In [ ]:
# 80 kW already installed — covers the 75 kW peak. With mandatory=False, optimizer skips.
result = build(
    boiler_size=Investment(
        min_size=50,
        max_size=200,
        mandatory=False,  # zero builds allowed
        prior_size=80,
        effects_per_size_at_build={'cost': 1500},
        effects_per_size_recurring={'cost': 30},
    ),
)
print(f'invest--size: {float(result.solution["invest--size"].sel(flow="boiler(heat)").values):.1f} kW')
result.solution['invest--build'].sel(flow='boiler(heat)').to_dataframe(name='built')

No new build in any period — the prior 80 kW covers everything. Drop `prior_size` to 40 (below peak) and a build appears immediately. Set `mandatory=True` and a build appears even when the prior is enough.

`invest--size` reports the *available* capacity — `prior_size` when no build, otherwise the chosen new size.


## Recap

| Field | Meaning |
|---|---|
| `effects_per_size_at_build` | Per-MW CAPEX, charged in the build period |
| `effects_fixed_at_build` | Fixed CAPEX, charged in the build period |
| `effects_per_size_recurring` | Per-MW recurring, every active period |
| `effects_fixed_recurring` | Fixed recurring, every active period |
| `lifetime` | Periods active after build; `None` = forever |
| `prior_size` | Pre-T0 capacity, no CAPEX |
| `mandatory` | True = build exactly once, False = at most once |

`Investment` builds on T7's multi-period model: same `periods=`, `period_weights=`, plus build-timing and a four-way at-build/recurring × per-size/fixed cost grid. One asset, one size, one CAPEX bill — closer to how real planning actually accounts.

Some scenarios are deliberately not covered here — they're what the current API can't yet express cleanly:

- **Replacement after retirement.** Each `Investment` is at-most-once, so once `lifetime` expires there's no rebuild. Tracked in [#102](https://github.com/FBumann/fluxopt/issues/102).
- **Installments and vintage-dependent costs.** Build in 2024, pay across 2024/2028/2032; or have an asset's recurring cost depend on when it was built. Both need a `(period, build_period)` cost matrix. Tracked in [#96](https://github.com/FBumann/fluxopt/issues/96), with broader scope in [#88](https://github.com/FBumann/fluxopt/issues/88).
- **`lifetime` semantics.** The unit (periods? years? counts including build?) needs clarification — see [#104](https://github.com/FBumann/fluxopt/issues/104).
